<a href="https://colab.research.google.com/github/aaronseymour7/MLQ/blob/main/uCJ.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install netket
!pip install qiskit
!pip install --upgrade numba netket
!pip install --upgrade pennylane
!pip install ffsim
!pip install qiskit-aer
!pip install qiskit-algorithms

In [ ]:
"""

Physical conventions
--------------------
  H = J Σ_i S·S_{i+1}   where  Sᵃ = σᵃ/2
    = J/4 Σ_i (XX + YY + ZZ)_{i,i+1}
  J = 1.0  (antiferromagnet)
  Qubit: |0⟩ = ↑, |1⟩ = ↓
"""

import numpy as np
from scipy.sparse import lil_matrix, csr_matrix
from scipy.sparse.linalg import eigsh

import netket as nk
import netket.nn as nknn
import flax.linen as nn
import jax.numpy as jnp
import jax
import optax

from qiskit import QuantumCircuit, QuantumRegister
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import Statevector, SparsePauliOp


# ═══════════════════════════════════════════════════════════════════════════
# PARAMETERS
# ═══════════════════════════════════════════════════════════════════════════

N           = 6
J_COUPLING  = 1.0
PBC         = True
ALPHA       = 3
VMC_SAMPLES = 512
VMC_STEPS   = 600
K_LAYERS    = 3
K_MAX       = 3
E_TOL       = 5e-3
SEED        = 23

# Even N: Sz = 0  (N/2 up-spins)
# Odd  N: Sz = +1/2  ((N+1)/2 up-spins)
N_QUBITS = N                                  # no ancilla — ever
N_UP     = N // 2 if N % 2 == 0 else (N + 1) // 2
TOTAL_SZ = 0 if N % 2 == 0 else 1            # NetKet total_sz (in units of 1/2? no — in units of S)
# NetKet total_sz is in units of ½: total_sz=0 → Sz=0, total_sz=1 → Sz=+½
NETKET_TOTAL_SZ = 0 if N % 2 == 0 else 1

#DIAGONALIZATION

def build_basis(n, n_up):
    return np.array([b for b in range(1 << n) if bin(b).count('1') == n_up],
                    dtype=np.int64)


def build_hamiltonian(n, n_up, j=1.0, pbc=True):
    basis   = build_basis(n, n_up)
    dim     = len(basis)
    idx_map = {int(b): i for i, b in enumerate(basis)}
    H       = lil_matrix((dim, dim), dtype=np.float64)
    edges   = ([(i, (i+1) % n) for i in range(n)] if pbc
               else [(i, i+1) for i in range(n-1)])

    for si, sj in edges:
        for row, bits in enumerate(basis):
            zi = 0.5 if (bits >> si) & 1 else -0.5
            zj = 0.5 if (bits >> sj) & 1 else -0.5
            H[row, row] += j * zi * zj                   # Sz·Sz
            if ((bits >> si) & 1) != ((bits >> sj) & 1): # S+S- + S-S+
                flipped = bits ^ (1 << si) ^ (1 << sj)
                col = idx_map.get(int(flipped), -1)
                if col >= 0:
                    H[row, col] += 0.5 * j
    return csr_matrix(H), basis, idx_map


print(f"\n{'='*60}")
print(f"  1D Heisenberg  N={N}  J={J_COUPLING}  PBC={PBC}")
print(f"  Sector: Sz = {'+' if N % 2 else ''}{NETKET_TOTAL_SZ}/2   N_up = {N_UP}")
print(f"{'='*60}")

BASIS, BINDEX = build_basis(N, N_UP), {}
H_SPARSE, BASIS, BINDEX = build_hamiltonian(N, N_UP, J_COUPLING, PBC)
DIM = len(BASIS)

evals, evecs = eigsh(H_SPARSE, k=1, which='SA')
E_EXACT   = float(evals[0])
PSI_EXACT = evecs[:, 0] / np.linalg.norm(evecs[:, 0])
print(f"[Lanczos]  E_exact = {E_EXACT:.8f}   E/site = {E_EXACT/N:.8f}")



class RBMModel(nn.Module):
    alpha: int = 1

    @nn.compact
    def __call__(self, x):
        x   = x.astype(jnp.complex128)
        a   = self.param('visible_bias', nn.initializers.normal(0.01),
                         (x.shape[-1],), jnp.complex128)
        W   = nn.Dense(self.alpha * x.shape[-1],
                       use_bias=True,
                       dtype=jnp.complex128,
                       param_dtype=jnp.complex128,
                       kernel_init=nn.initializers.normal(0.01),
                       bias_init=nn.initializers.normal(0.01))
        pre = W(x)
        return jnp.dot(x, a) + jnp.sum(nknn.log_cosh(pre), axis=-1)


def run_netket_vmc(n, j=1.0, pbc=True, alpha=1,
                   n_samples=512, n_steps=600, total_sz=0):
    hi  = nk.hilbert.Spin(s=0.5, N=n, total_sz=total_sz)
    # NetKet Heisenberg uses σ·σ (Pauli), not S·S (spin operators).
    # Since S = σ/2, we have S·S = σ·σ/4, so NetKet's J_nk = J/4
    # to match Lanczos where H = J sum S·S.
    ha  = nk.operator.Heisenberg(hilbert=hi,
                                  graph=nk.graph.Chain(n, pbc=pbc),
                                  J=j/4.0)
    sa  = nk.sampler.MetropolisExchange(hi, n_chains=16,
                                         graph=nk.graph.Chain(n))
    vs  = nk.vqs.MCState(sa, RBMModel(alpha=alpha),
                          n_samples=n_samples, seed=SEED)
    opt = optax.sgd(learning_rate=0.02)
    gs = nk.driver.VMC_SR(hamiltonian=ha, optimizer=opt, diag_shift=0.01, variational_state=vs)

    print(f"\n[NetKet VMC]  {n_steps} SR steps  "
          f"(N={n}, alpha={alpha}, samples={n_samples}, Sz={total_sz}/2)")
    gs.run(n_iter=n_steps, out=nk.logging.RuntimeLog())

    E_rbm = float(np.real(vs.expect(ha).mean))
    print(f"[NetKet VMC]  ⟨E⟩_RBM = {E_rbm:.8f}   E_exact = {E_EXACT:.8f}")
    ratio = E_rbm / E_EXACT if abs(E_EXACT) > 1e-10 else float('inf')
    if abs(ratio - 1.0) > 0.3:
        print(f"  WARNING: RBM/exact ratio = {ratio:.3f} — possible scale mismatch.")
    return vs, ha


vs_rbm, ha_netket = run_netket_vmc(N, J_COUPLING, PBC, ALPHA,
                                    VMC_SAMPLES, VMC_STEPS, NETKET_TOTAL_SZ)


# EXTRACT WARM-START CORRELATORS FROM RBM


def extract_rbm_correlators(vs, n, use_exact_basis=True):
    """
    Compute θ_J (Jastrow seed) and θ_K (orbital rotation seed) from RBM.


    use_exact_basis=True  : exact sum over full Sz sector  (default, N<=20)
    use_exact_basis=False : MC estimator fallback for very large N
    """


    # sigma must be shape (batch, N) with values in {-1, +1} (NetKet convention).
    def log_psi(sigma_batch):
        """Evaluate log ψ(σ) for a batch of configurations. Returns (batch,) complex."""
        return np.array(vs.log_value(jnp.array(sigma_batch, dtype=jnp.float32)))

    if use_exact_basis:
        # Full Sz-sector basis: (DIM, N), values ±1
        basis_arr = np.array(
            [[(1 if (b >> s) & 1 else -1) for s in range(n)] for b in BASIS],
            dtype=np.float32
        )   # (DIM, N)

        # All amplitudes in one call
        log_amps = log_psi(basis_arr)                             # (DIM,) complex
        # Normalise in log-space for numerical stability
        log_amps = log_amps - np.max(np.real(log_amps))
        amps     = np.exp(log_amps)                               # (DIM,) complex
        probs    = np.abs(amps) ** 2
        probs   /= probs.sum()                                    # normalise

        # ── Two-body correlators ───────────────────────────────────────────
        occ     = (basis_arr + 1) / 2                             # (DIM, N), 0/1
        n_mean  = (probs[:, None] * occ).sum(0)                   # (N,)
        nn_mean = np.einsum('d,di,dj->ij', probs, occ, occ)       # (N,N)
        C       = nn_mean - np.outer(n_mean, n_mean)

        # ── One-body density matrix ρ[i,j] = ⟨c†_i c_j⟩ ─────────────────
        # ρ[i,j] = Σ_σ |ψ(σ)|² · [ψ(σ^{j→i}) / ψ(σ)] · JW_sign(σ,i,j)
        # σ^{j→i}: annihilate j, create i  (only valid when i occ, j empty)
        rho = np.diag(n_mean.astype(complex))

        for i in range(n):
            for j in range(i + 1, n):
                mask = (basis_arr[:, i] == 1) & (basis_arr[:, j] == -1)
                if not mask.any():
                    continue

                sigma_v = basis_arr[mask]                         # (K, N)
                sigma_f = sigma_v.copy()
                sigma_f[:, i] = -1
                sigma_f[:, j] =  1

                # Single batched call for all valid configs (FIX A)
                lp_f = log_psi(sigma_f)                           # (K,) complex
                lp_v = log_psi(sigma_v)                           # (K,) complex
                ratio = np.exp(lp_f - lp_v)                       # ψ(flip)/ψ(σ)

                # Jordan-Wigner sign: parity of occupied sites strictly between i,j
                lo, hi = i, j
                jw_signs = np.array([
                    (-1) ** int(((sigma_v[k, lo+1:hi] + 1) / 2).sum())
                    for k in range(sigma_v.shape[0])
                ])

                rho[i, j] = (probs[mask] * jw_signs * ratio).sum()
                rho[j, i] = np.conj(rho[i, j])

    else:
        #MC fallback for large N
        vs.n_samples = 4096
        samples = np.array(vs.samples).reshape(-1, n)             # (S, N), ±1
        occ     = (samples + 1) / 2
        n_mean  = occ.mean(0)
        nn_mean = np.einsum('si,sj->ij', occ, occ) / len(samples)
        C       = nn_mean - np.outer(n_mean, n_mean)
        rho     = np.diag(n_mean.astype(complex))

        for i in range(n):
            for j in range(i + 1, n):
                mask = (samples[:, i] == 1) & (samples[:, j] == -1)
                if not mask.any():
                    continue
                sigma_v = samples[mask].astype(np.float32)
                sigma_f = sigma_v.copy()
                sigma_f[:, i] = -1;  sigma_f[:, j] = 1
                ratio = np.exp(log_psi(sigma_f) - log_psi(sigma_v))
                lo, hi = i, j
                jw_signs = np.array([
                    (-1) ** int(((sigma_v[k, lo+1:hi] + 1) / 2).sum())
                    for k in range(len(sigma_v))
                ])
                rho[i, j] = np.mean(jw_signs * ratio)
                rho[j, i] = np.conj(rho[i, j])

    # ── Map to circuit parameters ──────────────────────────────────────────
    alpha_J = 0.5
    alpha_K = 1.0

    theta_J = alpha_J * np.real(C)

    # K_gen = (ρ - ρ†)/2  is anti-Hermitian (purely imaginary off-diagonal).
    # np.imag(K_gen) gives real rotation angles for the Givens circuit.
    K_gen   = (rho - rho.conj().T) / 2.0
    theta_K = alpha_K * np.imag(K_gen)

    print(f"[Warm start]  |θ_J| max={np.max(np.abs(theta_J)):.4f}  "
          f"|θ_K| max={np.max(np.abs(theta_K)):.4f}")
    return theta_J, theta_K


theta_J_warm, theta_K_warm = extract_rbm_correlators(vs_rbm, N)

# ═══════════════════════════════════════════════════════════════════════════
# REFERENCE STATE — NÉEL  (correct sector per even/odd N)
# ═══════════════════════════════════════════════════════════════════════════
#
# Even N, Sz=0: N/2 up-spins on odd sites (0-indexed)
# Odd N, Sz=1/2: last site flipped to ↑
#                to achieve one extra up-spin.
#
# Qubit: |0⟩=↑, |1⟩=↓  → X gate on even sites (0,2,4,…) gives ↓ on even.
# Néel pattern: even sites ↓ (X gate), odd sites ↑ (no gate).

def add_neel_reference(qc, qreg, n):
    """
    Prepares the Néel state with correct N_UP = n//2 (even) or (n+1)//2 (odd).
    Even sites → |1⟩=↓  via X gate.
    For odd N the last qubit is left |0⟩=↑ (already contributing the extra ↑).
    """
    for i in range(n):
        if i % 2 == 0:   # even site → spin-down
            qc.x(qreg[i])


# ═══════════════════════════════════════════════════════════════════════════
#JAX STATEVECTOR ENGINE  (replaces Qiskit for optimization)
# ═══════════════════════════════════════════════════════════════════════════
#
#
#   Parameter-shift cost:  2 * n_params  forward passes per gradient
#   JAX autodiff cost:     1 forward + 1 backward pass  (≈2x total)
#
#   For k layers, n_params = 2k * N*(N-1)//2  → O(k*N²) per gradient call
#   Parameter-shift: O(k*N²) * O(2^N * N)
#   JAX grad:        O(2^N * N)
#
#


jax.config.update("jax_enable_x64", True)

# ── Precompute H in the Sz sector as scatter indices ──────────────────────

def build_jax_hamiltonian(n, n_up, j=1.0, pbc=True):
    """
    Returns (H_rows, H_cols, H_vals) as JAX int/float arrays.
    H|ψ⟩ = scatter_add(H_vals * ψ[H_cols], H_rows, dim=DIM)
    """
    basis_list = [b for b in range(1<<n) if bin(b).count('1')==n_up]
    idx_map    = {b:i for i,b in enumerate(basis_list)}
    rows, cols, vals = [], [], []
    edges = [(i,(i+1)%n) for i in range(n)] if pbc else [(i,i+1) for i in range(n-1)]
    for i,j_site in edges:
        for row, bits in enumerate(basis_list):
            zi = 0.5 if (bits>>i)&1 else -0.5
            zj = 0.5 if (bits>>j_site)&1 else -0.5
            rows.append(row); cols.append(row); vals.append(j * zi * zj)
            if ((bits>>i)&1) != ((bits>>j_site)&1):
                fl = bits ^ (1<<i) ^ (1<<j_site)
                if fl in idx_map:
                    rows.append(row); cols.append(idx_map[fl]); vals.append(0.5*j)
    return (jnp.array(rows, dtype=jnp.int32),
            jnp.array(cols, dtype=jnp.int32),
            jnp.array(vals, dtype=jnp.float64))

H_ROWS, H_COLS, H_VALS = build_jax_hamiltonian(N, N_UP, J_COUPLING, PBC)
DIM_JAX = len(BASIS)

@jax.jit
def apply_H_jax(psi):
    """Sparse H|ψ⟩ via scatter-add. Works with complex ψ."""
    return jnp.zeros(DIM_JAX, dtype=psi.dtype).at[H_ROWS].add(H_VALS * psi[H_COLS])

# ── Precompute Givens pair indices ─────────────────────────────────────────

def build_givens_pairs(n, basis, idx_map):
    """
    For each (i,j) pair, return (srcs, dsts) index arrays for the
    states where site i is occupied and site j is empty.
    These are static — only the angle theta changes.
    """
    pairs = []
    for i in range(n):
        for j in range(i+1, n):
            srcs, dsts = [], []
            for row, bits in enumerate(basis):
                if (bits>>i)&1 and not (bits>>j)&1:
                    fl = bits ^ (1<<i) ^ (1<<j)
                    if fl in idx_map:
                        srcs.append(row)
                        dsts.append(idx_map[fl])
            if srcs:
                pairs.append((jnp.array(srcs, dtype=jnp.int32),
                               jnp.array(dsts, dtype=jnp.int32)))
            else:
                pairs.append((jnp.array([], dtype=jnp.int32),
                               jnp.array([], dtype=jnp.int32)))
    return pairs

GIVENS_PAIRS = build_givens_pairs(N, list(BASIS), BINDEX)

def apply_givens_jax(psi, theta, srcs, dsts):
    """
    Number-conserving Givens rotation in the Sz sector.
    G(θ): psi[srcs] →  cos(θ)*psi[srcs] - sin(θ)*psi[dsts]
          psi[dsts] →  sin(θ)*psi[srcs] + cos(θ)*psi[dsts]
    """
    if len(srcs) == 0:
        return psi
    c, s    = jnp.cos(theta), jnp.sin(theta)
    p_s     = psi[srcs]
    p_d     = psi[dsts]
    return psi.at[srcs].set(c*p_s - s*p_d).at[dsts].set(s*p_s + c*p_d)

# ── Precompute Jastrow density-product matrix ──────────────────────────────

def build_jastrow_matrix(n, basis):
    """
    Returns NN_MAT of shape (DIM, n_pair) where NN_MAT[state, k] = n_i*n_j
    for the k-th pair (i,j). Used as: phases = NN_MAT @ theta_J
    """
    n_pair = n*(n-1)//2
    mat    = np.zeros((len(basis), n_pair))
    k = 0
    for i in range(n):
        for j in range(i+1, n):
            for row, bits in enumerate(basis):
                mat[row, k] = ((bits>>i)&1) * ((bits>>j)&1)
            k += 1
    return jnp.array(mat, dtype=jnp.float64)

NN_MAT = build_jastrow_matrix(N, list(BASIS))   # (DIM, n_pair)

@jax.jit
def apply_jastrow_jax(psi, theta_J):
    """exp(i J)|ψ⟩: multiply each amplitude by exp(i * NN_MAT @ theta_J)."""
    phases = NN_MAT @ theta_J
    return psi * jnp.exp(1j * phases)

# ── Néel reference state as JAX array ─────────────────────────────────────

def neel_jax(n, n_up, basis, idx_map):
    """
    Néel state in the Sz sector: even sites occupied (|1⟩=↓).
    For odd N this gives ceil(N/2) = N_UP up-spins. ✓
    """
    neel_bits = sum(1<<i for i in range(n) if i%2==0)
    assert neel_bits in idx_map, f"Neel state not in Sz sector (N={n}, N_up={n_up})"
    psi = jnp.zeros(len(basis), dtype=jnp.complex128)
    return psi.at[idx_map[neel_bits]].set(1.0)

PSI_NEEL = neel_jax(N, N_UP, list(BASIS), BINDEX)



def ucj_state_jax(theta, k_layers, psi0, n_pair, givens_pairs):
    """
    |Ψ(θ)⟩ = (exp(K) exp(J))^k |ref⟩  in JAX.
    theta layout: [tJ_0, tK_0, tJ_1, tK_1, ..., tJ_{k-1}, tK_{k-1}]
    Each tJ, tK has length n_pair.
    """
    psi = psi0.astype(jnp.complex128)
    for l in range(k_layers):
        tJ  = theta[l * 2*n_pair       : l * 2*n_pair + n_pair]
        tK  = theta[l * 2*n_pair + n_pair : (l+1) * 2*n_pair]
        psi = apply_jastrow_jax(psi, tJ)
        for p, (srcs, dsts) in enumerate(givens_pairs):
            psi = apply_givens_jax(psi, tK[p], srcs, dsts)
    return psi / jnp.linalg.norm(psi)

def energy_jax(theta, k_layers, psi0, n_pair, givens_pairs):
    """⟨Ψ(θ)|H|Ψ(θ)⟩ — differentiable through the full circuit."""
    psi = ucj_state_jax(theta, k_layers, psi0, n_pair, givens_pairs)
    return jnp.real(jnp.dot(jnp.conj(psi), apply_H_jax(psi)))

def make_jax_energy_and_grad(k_layers):
    """
    Returns (val_and_grad_fn, energy_fn) both JIT-compiled.
    val_and_grad_fn(theta) → (scalar_energy, gradient_array)
    Cost: 1 forward + 1 backward pass, independent of n_params.
    """
    import functools
    n_pair = N * (N-1) // 2
    _efn = functools.partial(energy_jax, k_layers=k_layers, psi0=PSI_NEEL,
                              n_pair=n_pair, givens_pairs=GIVENS_PAIRS)
    _vg  = jax.jit(jax.value_and_grad(_efn))
    _e   = jax.jit(_efn)
    return _vg, _e

# ── Fidelity (exact, post-optimization) ───────────────────────────────────

def compute_fidelity_jax(theta, k_layers, psi_exact):
    """|⟨ψ_exact|Ψ(θ)⟩|² computed in the Sz sector."""
    n_pair = N * (N-1) // 2
    psi    = ucj_state_jax(theta, k_layers, PSI_NEEL, n_pair, GIVENS_PAIRS)
    return float(jnp.abs(jnp.dot(jnp.conj(psi_exact.astype(jnp.complex128)), psi))**2)

# ═══════════════════════════════════════════════════════════════════════════
#QISKIT CIRCUIT  (visualization only )
# ═══════════════════════════════════════════════════════════════════════════

def add_neel_reference(qc, qreg, n):
    for i in range(n):
        if i % 2 == 0:
            qc.x(qreg[i])
    return qc

def _givens_gate(qc, qi, qj, theta):
    qc.cx(qj, qi)
    qc.ry(theta, qi)
    qc.cx(qi, qj)
    qc.ry(-theta, qi)
    qc.cx(qj, qi)

def build_visualization_circuit(n, k_layers, opt_theta):
    """
    Build a Qiskit circuit with the optimized parameters bound in,
    purely for visualization / export. Not used during optimization.
    """
    from qiskit import QuantumCircuit, QuantumRegister
    n_pair  = n * (n-1) // 2
    qreg    = QuantumRegister(n, 'q')
    qc      = QuantumCircuit(qreg)
    add_neel_reference(qc, qreg, n)
    for l in range(k_layers):
        qc.barrier(label=f'L{l}')
        tJ = opt_theta[l*2*n_pair       : l*2*n_pair + n_pair]
        tK = opt_theta[l*2*n_pair+n_pair : (l+1)*2*n_pair]
        # Jastrow: ZZ rotations + Z corrections (numeric params already bound)
        k = 0
        for i in range(n):
            for j in range(i+1, n):
                gamma = float(tJ[k]) / 4.0
                qc.cx(qreg[i], qreg[j])
                qc.rz(2.0*gamma, qreg[j])
                qc.cx(qreg[i], qreg[j])
                k += 1
        # Givens
        k = 0
        for i in range(n):
            for j in range(i+1, n):
                _givens_gate(qc, qreg[i], qreg[j], float(tK[k]))
                k += 1
    return qc

# ═══════════════════════════════════════════════════════════════════════════
# WARM-START PARAMETER VECTOR
# ═══════════════════════════════════════════════════════════════════════════

def build_warm_start(theta_J_mat, theta_K_mat, n, k_layers, seed=SEED):
    """
    Build initial parameter vector.

    theta_J: seeded from RBM connected correlator — good warm start.

    theta_K: the RBM gives Im(rho_ij)=0 for real-weight models, so
             theta_K_warm=0 exactly. The Neel state is a STATIONARY POINT
             of all Givens rotations at theta=0 (by particle-hole symmetry),
             so starting at theta_K=0 gives zero gradient and L-BFGS
             declares convergence immediately without moving.
             Fix: initialise theta_K with random uniform ~[-pi/4, pi/4]
             to break the symmetry and give nonzero gradients.
             The magnitude ~pi/4 is chosen to move well away from the
             degenerate point without overshooting the landscape.
    """
    n_pair = n * (n - 1) // 2
    rng    = np.random.default_rng(seed)

    def flatten_upper(mat):
        return np.array([mat[i, j]
                         for i in range(n) for j in range(i+1, n)])

    J_flat = flatten_upper(theta_J_mat)           # from RBM: good seed
    # K: ignore near-zero RBM value, use symmetry-breaking random init
    K_flat = rng.uniform(-np.pi/4, np.pi/4, n_pair)

    out = []
    for layer in range(k_layers):
        if layer == 0:
            out.append(J_flat)
            out.append(K_flat)
        else:
            out.append(0.05 * rng.standard_normal(n_pair))   # J
            out.append(rng.uniform(-np.pi/4, np.pi/4, n_pair))  # K
    return np.concatenate(out)

# ═══════════════════════════════════════════════════════════════════════════
# ENERGY + GRADIENT  (JAX autodiff — O(1) forward+backward passes)
# ═══════════════════════════════════════════════════════════════════════════



# ═══════════════════════════════════════════════════════════════════════════
# OPTIMISATION + ADAPTIVE K
# ═══════════════════════════════════════════════════════════════════════════

def optimize_layer(n, k, x0, maxiter=600):
    """
    Minimize ⟨H⟩ using JAX autodiff + scipy L-BFGS-B.
    val_and_grad computes energy and gradient in one forward+backward pass.
    Cost per L-BFGS step: O(2^N * N)  independent of n_params.
    """
    from scipy.optimize import minimize as scipy_minimize
    print(f"\n[uCJ k={k}]  {len(x0)} params,  maxiter={maxiter}")

    val_grad_fn, energy_fn = make_jax_energy_and_grad(k)

    # Warm up JIT compilation
    _ = val_grad_fn(jnp.array(x0, dtype=jnp.float64))

    best   = [np.inf]
    it     = [0]

    def scipy_fn(x_np):
        it[0] += 1
        x     = jnp.array(x_np, dtype=jnp.float64)
        E, g  = val_grad_fn(x)
        E_f   = float(E)
        if E_f < best[0]:
            best[0] = E_f
        if it[0] % 10 == 0:
            print(f"    iter {it[0]:4d}  E={E_f:.8f}"
                  f"  best={best[0]:.8f}  exact={E_EXACT:.8f}"
                  f"  ΔE={E_f-E_EXACT:+.2e}")
        return E_f, np.array(g, dtype=np.float64)

    E0 = float(energy_fn(jnp.array(x0, dtype=jnp.float64)))
    print(f"  x0 energy: {E0:.8f}")

    result   = scipy_minimize(scipy_fn, x0, jac=True, method='L-BFGS-B',
                               options={'maxiter': maxiter,
                                        'ftol': 1e-14, 'gtol': 1e-8})
    opt_x    = np.array(result.x)
    opt_E    = float(result.fun)
    fidelity = compute_fidelity_jax(jnp.array(opt_x), k, PSI_EXACT)

    print(f"  E={opt_E:.8f}  |⟨exact|uCJ⟩|²={fidelity:.6f}"
          f"  converged={result.success}  nit={result.nit}")
    return opt_x, opt_E, fidelity

N_RESTARTS = 3   # random restarts per k-layer to escape local minima

def adaptive_ucj(n, k_init, k_max, e_tol, theta_J_warm, theta_K_warm,
                 maxiter=300, n_restarts=N_RESTARTS):
    """
    Adaptive k with multi-restart per layer.
    Multi-restart is essential because the Neel→uCJ landscape has many
    local minima depending on the random K initialisation direction.
    """
    best = dict(E=np.inf, fid=0., params=None, k=k_init)
    prev_params = None

    for k in range(k_init, k_max + 1):
        n_pair = n * (n - 1) // 2
        n_per  = 2 * n_pair
        layer_best = dict(E=np.inf, params=None, fid=0.)

        for restart in range(n_restarts):
            print(f"\n  [k={k}, restart={restart+1}/{n_restarts}]")
            if prev_params is None:
                x0 = build_warm_start(theta_J_warm, theta_K_warm, n, k,
                                       seed=SEED + restart)
            else:
                # Append a new symmetry-breaking layer to previous optimised params
                rng       = np.random.default_rng(SEED + k * 100 + restart)
                new_layer = np.concatenate([
                    0.05 * rng.standard_normal(n_pair),          # J
                    rng.uniform(-np.pi/4, np.pi/4, n_pair)       # K
                ])
                x0 = np.concatenate([prev_params, new_layer])

            opt_x, opt_E, fid = optimize_layer(n, k, x0, maxiter)

            if opt_E < layer_best['E']:
                layer_best.update(E=opt_E, params=opt_x, fid=fid)

        # Keep best result from all restarts for this layer
        opt_x, opt_E, fid = layer_best['params'], layer_best['E'], layer_best['fid']
        prev_params = opt_x

        if opt_E < best['E']:
            best.update(E=opt_E, fid=fid, params=opt_x, k=k)

        delta = abs(opt_E - E_EXACT)
        print(f"\n  [k={k}]  best E={opt_E:.8f}  |ΔE|={delta:.4e}  (tol={e_tol:.4e})")
        if delta < e_tol:
            print(f"  Converged at k={k}.")
            break
        elif k < k_max:
            print(f"  Adding layer {k+1}…")

    return best


# ═══════════════════════════════════════════════════════════════════════════
# MAIN
# ═══════════════════════════════════════════════════════════════════════════

if __name__ == '__main__':

    best = adaptive_ucj(N, K_LAYERS, K_MAX, E_TOL,
                        theta_J_warm, theta_K_warm, maxiter=600)

    print(f"\n{'='*60}")
    print(f"  FINAL RESULTS  (N={N}, PBC={PBC})")
    print(f"{'='*60}")
    print(f"  Sector            :  Sz = {NETKET_TOTAL_SZ}/2")
    print(f"  E_exact           =  {E_EXACT:.10f}  ({E_EXACT/N:.8f}/site)")
    print(f"  E_uCJ  (k={best['k']})    =  {best['E']:.10f}  ({best['E']/N:.8f}/site)")
    print(f"  |ΔE|              =  {abs(best['E']-E_EXACT):.4e}")
    print(f"  Rel. error        =  {abs(best['E']-E_EXACT)/abs(E_EXACT):.4e}")
    print(f"  |⟨exact|uCJ⟩|²   =  {best['fid']:.8f}")
    print(f"{'='*60}")

    # Build Qiskit circuit for visualization only
    try:
        from qiskit import QuantumCircuit, QuantumRegister
        qc_vis = build_visualization_circuit(N, best['k'],
                                             jnp.array(best['params']))
        print(f"\n  Circuit depth = {qc_vis.depth()}   Gates = {qc_vis.count_ops()}")
        #print(qc_vis.draw(fold=120))
    except ImportError:
        print("  (Qiskit not available for circuit diagram)")